# 02_01 · IM：无未来泄露的多窗口滚动 BPV 日内季节因子

读取 `01_01_im_main_continuous.ipynb` 生成的 `im_main_continuous_1m.parquet`。使用实际合约的**未复权 close**；主力选择沿用上游前一交易日信号。此处只读源文件和正式交易日日历，结果留在当前研究目录。使用 **latitude_env_v2** 内核，从头运行。

$$r_{d,m}=\log P_{d,m}-\log P_{d,m-1},\quad b_{d,m}=\frac{\pi}{2}|r_{d,m}||r_{d,m-1}|$$

$b$ 是 **minute BPV contribution**；BPV 的跳跃鲁棒性来自多个 contribution 的聚合，不把单个 contribution 称为单分钟 BPV estimator。

$$B_d=\sum_m b_{d,m},\quad N_d=\#\{m:b_{d,m}\text{ 有效}\},\quad u_{d,m}=N_db_{d,m}/B_d$$

$$\hat s_{d,m}=\operatorname{median}_{j=d-L,\ldots,d-1}u_{j,m},\quad
\hat s_{d,m}\leftarrow\hat s_{d,m}/\operatorname{mean}_m\hat s_{d,m}$$

**分别输出 30、60、120、360 个交易日窗口**，记窗口为 $L$。先按完整交易日 `shift(1)` 再 rolling。每个窗口的前 $L$ 个交易日保持 NaN，不使用 expanding，不拟合全样本静态曲线。每个目标日只用开盘前已结束的历史日；历史日使用完整日 $B_j$ 合法，$u_d$ 本身只是收盘后中间量，不能作为当天可用特征。

输入投影：`contract_code: string`、`trading_date: date32`、`session_number: int8`、`bar_at: timestamp[us, Asia/Shanghai]`、`close: float64`、`signal_trading_date: date32`。输出主键为 `trade_date + minute_slot`；`minute_slot` 是北京时间的分钟末端时刻 `time64[us]`。开盘前曲线以及输出的局部 Arrow Schema 均在下方显式呈现。

**阶段阅读导航**：[01_01 主力连续价格](01_01_im_main_continuous.ipynb) → [02_01 BPV季节因子](02_01_im_bpv_intraday_seasonality.ipynb)（[02_02 窗口比较](02_02_im_bpv_window_comparison.ipynb)）→ [03_01 采样时钟与价格](03_01_im_clock_resampling.ipynb)（[03_02 时钟诊断](03_02_im_clock_comparison_diagnostics.ipynb)）→ [04_01 波动观测量](04_01_im_volatility_observations.ipynb) → [04_02 每日拟合](04_02_im_volatility_model_fitting.ipynb) → [04_03 下一采样步预测](04_03_im_next_step_volatility_forecasting.ipynb)／[04_04 未来15分钟预测](04_04_im_15m_volatility_forecasting.ipynb)。括号中的比较与诊断本用于查看对应阶段结果，不是下一阶段的数据生成依赖；最后两本预测相互独立。


In [1]:
import pathlib
import sys
import datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import hashlib
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")
from config.settings import settings
from config.data_contracts import (
    TRADE_CALENDAR_SCHEMA, validate_arrow_table, pandas_to_arrow, arrow_to_pandas,
)

research_dir = candidate_root / "01_project_collection/JQ_strategy/draft_experiments"
source_path = research_dir / "im_main_continuous_1m.parquet"
lookbacks = (30, 60, 120, 360)
source_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
as_of_at = pd.Timestamp.now(tz="Asia/Shanghai")
print("Python:", sys.executable)
print("本次数据可见截止时间:", as_of_at)
print("窗口（交易日）:", lookbacks)

Python: E:\anaconda3\envs\latitude\python.exe
本次数据可见截止时间: 2026-09-06 00:16:03.892869+08:00
窗口（交易日）: (30, 60, 120, 360)


## 1. 读取连续序列和交易日日历

不使用复权价格，不重新选择主力。输入必须有实际合约代码和严格早于目标日的选择信号。
下方日历仅用于对齐交易日，避免整日分钟缺失后把 交易日窗口偷偷压缩；不读取任何当天价格来确定历史曲线。源文件范围以外不额外生成预测日期。

In [2]:
input_schema = pa.schema([
    pa.field("contract_code", pa.string(), False),
    pa.field("trading_date", pa.date32(), False),
    pa.field("session_number", pa.int8(), False),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("close", pa.float64(), False),
    pa.field("signal_trading_date", pa.date32(), False),
])
input_table = pq.read_table(source_path, columns=input_schema.names)
if not input_table.schema.equals(input_schema, check_metadata=False):
    raise TypeError("连续序列输入的物理 Schema 不符合本实验契约")
minute_df = arrow_to_pandas(input_table, input_schema)
minute_df = minute_df.rename(columns={"trading_date": "trade_date"})
minute_df["trade_date"] = pd.to_datetime(minute_df["trade_date"]).dt.date
minute_df["signal_trading_date"] = pd.to_datetime(minute_df["signal_trading_date"]).dt.date
minute_df["bar_at"] = pd.to_datetime(minute_df["bar_at"])
minute_df["close"] = minute_df["close"].astype("float64")
minute_df = minute_df.sort_values(["trade_date", "bar_at"]).reset_index(drop=True)
if minute_df.empty:
    raise ValueError("连续分钟序列为空")
if not minute_df["contract_code"].str.fullmatch(r"IM[0-9]{4}\.CCFX").all():
    raise ValueError("必须使用 IM 实际月份合约")
if not (minute_df["signal_trading_date"] < minute_df["trade_date"]).all():
    raise ValueError("主力选择信号必须严格早于目标交易日")
if (minute_df["bar_at"] > as_of_at).any():
    raise ValueError("存在本次数据可见截止时间之后的行情")
if (minute_df.groupby("trade_date")["contract_code"].nunique() != 1).any():
    raise ValueError("上游日级主力映射必须在开盘前确定，每天只有一个实际合约")
display(minute_df.head())

,contract_code,trade_date,session_number,bar_at,close,signal_trading_date
0,IM2208.CCFX,2022-07-25,1,2022-07-25 09:31:00+08:00,6973.8,2022-07-22
1,IM2208.CCFX,2022-07-25,1,2022-07-25 09:32:00+08:00,6982.8,2022-07-22
2,IM2208.CCFX,2022-07-25,1,2022-07-25 09:33:00+08:00,6989.8,2022-07-22
3,IM2208.CCFX,2022-07-25,1,2022-07-25 09:34:00+08:00,6993.0,2022-07-22
4,IM2208.CCFX,2022-07-25,1,2022-07-25 09:35:00+08:00,7002.2,2022-07-22


In [3]:
calendar_schema = TRADE_CALENDAR_SCHEMA
calendar_table_name = calendar_schema.metadata[b"table_name"].decode("utf-8")
calendar_partition_columns = calendar_schema.metadata[b"partition_columns"].decode("utf-8").split(",")
calendar_partitioning = ds.partitioning(
    pa.schema([calendar_schema.field(name) for name in calendar_partition_columns]), flavor="hive",
)
calendar_projection_schema = pa.schema([
    calendar_schema.field("calendar_date"), calendar_schema.field("is_trading_day"),
])
print(calendar_table_name, "主键:", calendar_schema.metadata[b"primary_key"].decode("utf-8"))
print(calendar_projection_schema)
calendar_dataset = ds.dataset(
    pathlib.Path(settings.futures_lake_root) / "silver" / calendar_table_name,
    format="parquet", partitioning=calendar_partitioning,
)
for field in calendar_projection_schema:
    if not calendar_dataset.schema.field(field.name).equals(field, check_metadata=False):
        raise TypeError(f"交易日日历物理字段不兼容: {field.name}")
calendar_table = calendar_dataset.to_table(
    columns=calendar_projection_schema.names,
    filter=(ds.field("calendar_date") >= minute_df["trade_date"].min())
           & (ds.field("calendar_date") <= minute_df["trade_date"].max())
           & ds.field("is_trading_day"),
)
calendar_df = arrow_to_pandas(calendar_table, calendar_projection_schema)
trade_dates = pd.Index(sorted(pd.to_datetime(calendar_df["calendar_date"]).dt.date), name="trade_date")
if not minute_df["trade_date"].isin(trade_dates).all():
    raise ValueError("源文件日期超出正式交易日日历")
print(f"{len(minute_df):,} 根分钟线，{len(trade_dates)} 个交易日")

dim_trade_calendar 主键: calendar_date
calendar_date: date32[day] not null
  -- field metadata --
  field_name_zh: '日历日期'
  description_zh: '日历日期；本行代表的中国自然日。'
  source_system_zh: '系统生成'
  source_api: '系统生成'
  source_column: '—'
  transformation_zh: '在起止日之间逐日生成；单位：日；角' + 24
  unit_zh: '日'
  nullable_reason_zh: '不可空、唯一、严格逐日连续。'
  semantic_role_zh: '主键、时间'
  enum_values_zh: '非枚举'
  quality_rules_zh: '不可空、唯一、严格逐日连续。'
is_trading_day: bool not null
  -- field metadata --
  field_name_zh: '该自然日是否为中国期货交易日'
  description_zh: '该自然日是否为中国期货交易日。'
  source_system_zh: 'JQData get_trade_days'
  source_api: 'JQData get_trade_days'
  source_column: '返回日期元素'
  transformation_zh: '日期是否属于 API 返回集合；角色：日�' + 11
  unit_zh: '不适用'
  nullable_reason_zh: '不可空；允许值：true=交易日、false=非' + 12
  semantic_role_zh: '日历状态'
  enum_values_zh: '非枚举'
  quality_rules_zh: '不可空；允许值：true=交易日、false=非�' + 10
238,800 根分钟线，995 个交易日


## 2. 合约内收益与 minute BPV contribution

本实验固定支持现有 IM 分钟末端标签：09:31–11:30、13:01–15:00，共 240 个位置。时间结构不同的输入直接报错，不从全样本推断 slot 集合。缺失行重建为空格，不 forward-fill。

先在 `trade_date + contract_code + session_number` 内计算收益及 contribution，再按时间拼接。收益要求真实时间差恰好为一分钟；相邻两个收益必须来自同一段连续行情。因此正常日 09:31、09:32、13:01、13:02 的 contribution 均为 NaN，$N_d=236$。如果更换输入只剩连续价格，也必须保留可靠的合约/换月边界标识，否则无法证明没有跨合约收益，本入口拒绝这种输入。

In [4]:
minute_slots = pd.Index([
    *pd.date_range("2000-01-01 09:31", "2000-01-01 11:30", freq="min").time,
    *pd.date_range("2000-01-01 13:01", "2000-01-01 15:00", freq="min").time,
], name="minute_slot")

def compute_minute_bpv(contract_minute_df):
    """在实际合约的连续一分钟区间内计算收益与 BPV contribution。"""
    bpv_df = contract_minute_df.copy().sort_values(
        ["trade_date", "contract_code", "session_number", "bar_at"],
    ).reset_index(drop=True)
    bpv_df["minute_slot"] = bpv_df["bar_at"].dt.time
    if bpv_df.duplicated(["trade_date", "minute_slot"]).any():
        raise ValueError("连续序列的交易日 + minute slot 必须唯一")
    if not bpv_df["minute_slot"].isin(minute_slots).all():
        raise ValueError("不支持的 IM 时间结构或非分钟末端标签")
    if not (bpv_df["bar_at"].dt.date == bpv_df["trade_date"]).all():
        raise ValueError("IM 日盘时间戳与交易日不一致")
    expected_session = np.where(bpv_df["minute_slot"] <= dt.time(11, 30), 1, 2)
    if not (bpv_df["session_number"] == expected_session).all():
        raise ValueError("IM Session 归属与当前支持的时间结构不一致")
    bpv_df["is_valid_price"] = np.isfinite(bpv_df["close"]) & (bpv_df["close"] > 0)
    bpv_df["log_close"] = np.log(bpv_df["close"].where(bpv_df["is_valid_price"]))
    contract_groups = bpv_df.groupby(["trade_date", "contract_code", "session_number"], sort=False)
    consecutive_minute = contract_groups["bar_at"].diff().eq(pd.Timedelta(minutes=1))
    bpv_df["log_return"] = contract_groups["log_close"].diff().where(consecutive_minute)
    previous_return = contract_groups["log_return"].shift(1)
    bpv_df["bpv_contribution"] = (np.pi / 2) * bpv_df["log_return"].abs() * previous_return.abs()
    return bpv_df.sort_values(["trade_date", "bar_at"]).reset_index(drop=True)

bpv_df = compute_minute_bpv(minute_df)
bpv_matrix = bpv_df.pivot(index="trade_date", columns="minute_slot", values="bpv_contribution").reindex(
    index=trade_dates, columns=minute_slots,
)
price_coverage_matrix = bpv_df.pivot(index="trade_date", columns="minute_slot", values="is_valid_price").reindex(
    index=trade_dates, columns=minute_slots,
).fillna(False).astype(bool)
# 完整日要求时间已经结束，且所有 240 个预期价格位置有效。
# 本实验保守地只把 as_of_at 之前的自然日作为已结束日，不使用运行当天的形状。
completed_day_series = pd.Series(trade_dates < as_of_at.date(), index=trade_dates)
completed_day_series &= price_coverage_matrix.all(axis=1)
print("完整历史日:", int(completed_day_series.sum()), "缺失/未结束日:", int((~completed_day_series).sum()))
display(bpv_df[["trade_date", "minute_slot", "contract_code", "log_return", "bpv_contribution"]].head())

完整历史日: 995 缺失/未结束日: 0


,trade_date,minute_slot,contract_code,log_return,bpv_contribution
0,2022-07-25,09:31:00,IM2208.CCFX,NaN,NaN
1,2022-07-25,09:32:00,IM2208.CCFX,0.001290,NaN
2,2022-07-25,09:33:00,IM2208.CCFX,0.001002,2.029849e-06
3,2022-07-25,09:34:00,IM2208.CCFX,0.000458,7.203716e-07
4,2022-07-25,09:35:00,IM2208.CCFX,0.001315,9.452452e-07


## 3. 日级形状与严格多窗口交易日滚动

完整日内先用有效 contribution 数 $N_d$ 和日总和 $B_d$ 去除整体波动水平。$B_d\leq0$、非有限值或未完整结束的日期不产生 $u_d$。

固定窗口按正式交易日推进，不删除缺失日期、不向更早日期借样本。过去 $L$ 个交易日必须全部是可用完整日；每个 slot 也要求 $L$ 个有效历史值（`min_periods=lookback`）。常年无定义的边界 slot 保持 NaN。此保守缺失策略可能让缺失发生后若干日无曲线，避免静默改变窗口定义。

归一化分母只用当天**历史滚动曲线**的有限 slot，绝不按目标日实际有哪些价格或 contribution 重算分母。中位数为 0 的 slot 保留因子 0，对应的除法输出 NaN，不加任意 epsilon。

In [5]:
def estimate_bpv_seasonality(bpv_matrix, completed_day_series, lookback=60):
    """完整日相对形状及按交易日滞后一日的固定窗口季节曲线。"""
    if not isinstance(lookback, int) or lookback < 1:
        raise ValueError("lookback 必须为正整数交易日数")
    daily_bpv = bpv_matrix.sum(axis=1, min_count=1)
    valid_slot_count = bpv_matrix.count(axis=1)
    eligible_day = completed_day_series.reindex(bpv_matrix.index, fill_value=False)
    eligible_day = eligible_day & np.isfinite(daily_bpv) & daily_bpv.gt(0)
    daily_shape_matrix = bpv_matrix.mul(valid_slot_count, axis=0).div(daily_bpv, axis=0)
    daily_shape_matrix.loc[~eligible_day, :] = np.nan

    # 整天 shift，先将 d 日数据移到 d+1，再计算固定 lookback 个交易日窗口。
    lagged_shape_matrix = daily_shape_matrix.shift(1)
    rolling_median_matrix = lagged_shape_matrix.rolling(window=lookback, min_periods=lookback).median()
    history_day_count = eligible_day.astype("int64").shift(1).rolling(
        window=lookback, min_periods=lookback,
    ).sum()
    rolling_median_matrix.loc[history_day_count.ne(lookback), :] = np.nan
    curve_mean = rolling_median_matrix.mean(axis=1)
    seasonality_matrix = rolling_median_matrix.div(curve_mean.where(curve_mean.gt(0)), axis=0)
    daily_summary_df = pd.DataFrame({
        "daily_bpv": daily_bpv,
        "valid_slot_count": valid_slot_count,
        "is_eligible_history_day": eligible_day,
        "history_day_count": history_day_count,
    })
    return daily_shape_matrix, seasonality_matrix, daily_summary_df


seasonality_by_lookback = {}
daily_summary_by_lookback = {}
history_end_dates = pd.Series(trade_dates, index=trade_dates).shift(1)
for lookback in lookbacks:
    daily_shape_matrix, seasonality_matrix, daily_summary_df = estimate_bpv_seasonality(
        bpv_matrix, completed_day_series, lookback,
    )
    daily_summary_df["history_end_date"] = history_end_dates.where(seasonality_matrix.notna().any(axis=1))
    seasonality_by_lookback[lookback] = seasonality_matrix
    daily_summary_by_lookback[lookback] = daily_summary_df
    print(f"窗口 {lookback} 个交易日；可用曲线 {seasonality_matrix.notna().any(axis=1).sum()} 天")


窗口 30 个交易日；可用曲线 965 天
窗口 60 个交易日；可用曲线 935 天


窗口 120 个交易日；可用曲线 875 天


窗口 360 个交易日；可用曲线 635 天


## 4. 分钟输出

$$b^{adj}_{d,m}=b_{d,m}/\hat s_{d,m},\qquad f_{d,m}=\sqrt{\hat s_{d,m}},\qquad r^{adj}_{d,m}=r_{d,m}/f_{d,m}$$

`bpv_seasonality` 位于 **variance/BPV scale**；`volatility_seasonality` 位于 volatility scale。收益不能直接除以 BPV 因子。`bpv_adjusted` 是原 contribution 的直接去季节值，不等同于重新用调整后的相邻收益计算 BPV。

缺失分钟保留输出行，其 contribution 为 NaN；若历史足够，其开盘前已知的季节因子仍保留。由于收益和 BPV 的定义域不同，09:32、13:02 虽可能有收益，但没有 BPV 季节因子，调整收益也保留 NaN。

In [6]:

output_index = pd.MultiIndex.from_product([trade_dates, minute_slots], names=["trade_date", "minute_slot"])
base_output_df = bpv_df.set_index(["trade_date", "minute_slot"])[
    ["contract_code", "bar_at", "log_return", "bpv_contribution"]
].reindex(output_index)
output_by_lookback = {}
output_schema_by_lookback = {}
for lookback in lookbacks:
    output_df = base_output_df.copy()
    output_df["bpv_seasonality"] = seasonality_by_lookback[lookback].to_numpy(dtype="float64").reshape(-1)
    positive_seasonality = output_df["bpv_seasonality"].where(output_df["bpv_seasonality"].gt(0))
    output_df["bpv_adjusted"] = output_df["bpv_contribution"] / positive_seasonality
    output_df["volatility_seasonality"] = np.sqrt(output_df["bpv_seasonality"])
    output_df["return_adjusted"] = output_df["log_return"] / np.sqrt(positive_seasonality)
    output_df = output_df.reset_index()
    output_df["lookback_trading_days"] = lookback
    output_df["history_end_date"] = output_df["trade_date"].map(daily_summary_by_lookback[lookback]["history_end_date"])
    output_schema = pa.schema([
        pa.field("trade_date", pa.date32(), False),
        pa.field("minute_slot", pa.time64("us"), False),
        pa.field("lookback_trading_days", pa.int16(), False),
        pa.field("bpv_contribution", pa.float64()),
        pa.field("bpv_seasonality", pa.float64()),
        pa.field("bpv_adjusted", pa.float64()),
        pa.field("contract_code", pa.string()),
        pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai")),
        pa.field("log_return", pa.float64()),
        pa.field("volatility_seasonality", pa.float64()),
        pa.field("return_adjusted", pa.float64()),
        pa.field("history_end_date", pa.date32()),
    ], metadata={
        b"primary_key": b"trade_date,minute_slot",
        b"lookback_trading_days": str(lookback).encode(),
        b"seasonality_scale": b"variance/BPV",
        b"estimation": b"daily normalization; shift(1); rolling median; slot mean normalization",
        b"missing_policy": f"all {lookback} prior days complete; {lookback} valid observations per slot; no fill".encode(),
        b"as_of_at": as_of_at.isoformat().encode(),
        b"source_file": source_path.name.encode(),
        b"source_sha256": source_sha256.encode(),
    })

    output_by_lookback[lookback] = output_df[output_schema.names]
    output_schema_by_lookback[lookback] = output_schema
print(output_schema_by_lookback[lookbacks[0]])
display(output_by_lookback[lookbacks[0]].tail())


trade_date: date32[day] not null
minute_slot: time64[us] not null
lookback_trading_days: int16 not null
bpv_contribution: double
bpv_seasonality: double
bpv_adjusted: double
contract_code: string
bar_at: timestamp[us, tz=Asia/Shanghai]
log_return: double
volatility_seasonality: double
return_adjusted: double
history_end_date: date32[day]
-- schema metadata --
primary_key: 'trade_date,minute_slot'
lookback_trading_days: '30'
seasonality_scale: 'variance/BPV'
estimation: 'daily normalization; shift(1); rolling median; slot mean nor' + 10
missing_policy: 'all 30 prior days complete; 30 valid observations per sl' + 11
as_of_at: '2026-09-06T00:16:03.892869+08:00'
source_file: 'im_main_continuous_1m.parquet'
source_sha256: 'f64428741ab14de067ac06e8d9e1cd83a39f985f25a02a20898651430' + 7


,trade_date,minute_slot,lookback_trading_days,bpv_contribution,bpv_seasonality,bpv_adjusted,contract_code,bar_at,log_return,volatility_seasonality,return_adjusted,history_end_date
238795,2026-08-28,14:56:00,30,3.212797e-08,0.181302,1.772069e-07,IM2609.CCFX,2026-08-28 14:56:00+08:00,0.000392,0.425796,0.000920,2026-08-27
238796,2026-08-28,14:57:00,30,4.817623e-08,0.152826,3.152356e-07,IM2609.CCFX,2026-08-28 14:57:00+08:00,-0.000078,0.390930,-0.000200,2026-08-27
238797,2026-08-28,14:58:00,30,5.779111e-08,0.265297,2.178352e-07,IM2609.CCFX,2026-08-28 14:58:00+08:00,0.000470,0.515070,0.000912,2026-08-27
238798,2026-08-28,14:59:00,30,1.925566e-08,0.397442,4.844896e-08,IM2609.CCFX,2026-08-28 14:59:00+08:00,-0.000026,0.630430,-0.000041,2026-08-27
238799,2026-08-28,15:00:00,30,1.069396e-08,0.304013,3.517605e-08,IM2609.CCFX,2026-08-28 15:00:00+08:00,0.000261,0.551373,0.000473,2026-08-27


## 5. 防泄露与边界验证

逐窗口检查预热期、日均归一化、已知历史截止日。独立逐日仅取前 $L$ 个交易日中位数，与向量实现比对；再修改目标日及所有未来原始价格、截掉未来、删掉目标日部分分钟，要求目标日及更早曲线不变。额外用人工缺口、无效价格和盘中合约边界检查两个受影响的 contribution，以及缺失历史日不会被跳过。

In [7]:

for lookback in lookbacks:
    seasonality_matrix = seasonality_by_lookback[lookback]
    daily_summary_df = daily_summary_by_lookback[lookback]
    output_df = output_by_lookback[lookback]
    assert seasonality_matrix.iloc[:lookback].isna().all().all()
    np.testing.assert_allclose(daily_shape_matrix.mean(axis=1).dropna(), 1.0, rtol=1e-12)
    np.testing.assert_allclose(seasonality_matrix.mean(axis=1).dropna(), 1.0, rtol=1e-12)
    assert output_df.loc[output_df["bpv_seasonality"].notna(), "history_end_date"].lt(
        output_df.loc[output_df["bpv_seasonality"].notna(), "trade_date"]
    ).all()
    assert not output_df.duplicated(["trade_date", "minute_slot"]).any()
    boundary_slots = [dt.time(9, 31), dt.time(9, 32), dt.time(13, 1), dt.time(13, 2)]
    assert bpv_matrix[boundary_slots].isna().all().all()
    assert seasonality_matrix[boundary_slots].isna().all().all()

    # 独立逐日参考：只截取 [d-lookback, d)，不调用 rolling。
    for day_number in range(lookback, len(trade_dates)):
        history_matrix = daily_shape_matrix.iloc[day_number-lookback:day_number]
        if daily_summary_df["is_eligible_history_day"].iloc[day_number-lookback:day_number].all():
            expected_curve = history_matrix.median(axis=0).where(history_matrix.count().eq(lookback))
            expected_mean = expected_curve.mean()
            expected_curve = expected_curve / expected_mean if expected_mean > 0 else expected_curve * np.nan
        else:
            expected_curve = pd.Series(np.nan, index=minute_slots)
        np.testing.assert_allclose(seasonality_matrix.iloc[day_number], expected_curve, equal_nan=True, rtol=1e-12)

    if len(trade_dates) > lookback:
        target_day = trade_dates[max(lookback, len(trade_dates) // 2)]
        changed_minute_df = minute_df.copy()
        changed_rows = changed_minute_df["trade_date"] >= target_day
        changed_minute_df.loc[changed_rows, "close"] *= np.exp(
            0.15 * np.sin(np.arange(int(changed_rows.sum())) * 0.7)
        )
        changed_bpv_df = compute_minute_bpv(changed_minute_df)
        changed_bpv_matrix = changed_bpv_df.pivot(index="trade_date", columns="minute_slot", values="bpv_contribution").reindex(
            index=trade_dates, columns=minute_slots,
        )
        _, changed_seasonality_matrix, _ = estimate_bpv_seasonality(changed_bpv_matrix, completed_day_series, lookback)
        pd.testing.assert_frame_equal(seasonality_matrix.loc[:target_day], changed_seasonality_matrix.loc[:target_day])
        _, prefix_seasonality_matrix, _ = estimate_bpv_seasonality(
            bpv_matrix.loc[:target_day], completed_day_series.loc[:target_day], lookback,
        )
        pd.testing.assert_frame_equal(seasonality_matrix.loc[:target_day], prefix_seasonality_matrix)
        missing_target_matrix = bpv_matrix.copy()
        missing_target_matrix.loc[target_day, minute_slots[40:80]] = np.nan
        missing_target_completed = completed_day_series.copy()
        missing_target_completed.loc[target_day] = False
        _, missing_target_seasonality, _ = estimate_bpv_seasonality(missing_target_matrix, missing_target_completed, lookback)
        pd.testing.assert_frame_equal(seasonality_matrix.loc[:target_day], missing_target_seasonality.loc[:target_day])

    # 选一个真实完整日构造边界反例，验证缺口/坏价/切换不会制造普通一分钟收益。
    probe_day = completed_day_series[completed_day_series].index[0]
    probe_minute_df = minute_df.loc[minute_df["trade_date"] == probe_day].copy().reset_index(drop=True)
    for boundary_kind in ["gap", "invalid_price", "contract_change"]:
        boundary_minute_df = probe_minute_df.copy()
        boundary_at = boundary_minute_df.loc[30, "bar_at"]
        if boundary_kind == "gap":
            boundary_minute_df = boundary_minute_df.drop(index=30)
            affected_at = [boundary_at + pd.Timedelta(minutes=1), boundary_at + pd.Timedelta(minutes=2)]
        elif boundary_kind == "invalid_price":
            boundary_minute_df.loc[30, "close"] = 0.0
            affected_at = [boundary_at + pd.Timedelta(minutes=i) for i in range(3)]
        else:
            boundary_minute_df.loc[30:, "contract_code"] = "IM9912.CCFX"
            affected_at = [boundary_at, boundary_at + pd.Timedelta(minutes=1)]
        boundary_bpv_df = compute_minute_bpv(boundary_minute_df).set_index("bar_at")
        assert boundary_bpv_df.loc[affected_at, "bpv_contribution"].isna().all(), boundary_kind

    if len(trade_dates) > lookback:
        incomplete_history = completed_day_series.copy()
        incomplete_history.iloc[lookback - 1] = False
        _, incomplete_seasonality, _ = estimate_bpv_seasonality(bpv_matrix, incomplete_history, lookback)
        assert incomplete_seasonality.iloc[lookback:2 * lookback].isna().all().all()
        zero_bpv_matrix = bpv_matrix.copy()
        zero_bpv_matrix.iloc[lookback - 1] = 0.0
        _, zero_seasonality, _ = estimate_bpv_seasonality(zero_bpv_matrix, completed_day_series, lookback)
        assert zero_seasonality.iloc[lookback:2 * lookback].isna().all().all()
    print(f"窗口 {lookback} 交易日通过：预热、归一化、逐日参考、未来扰动、截断不变、目标日缺失及边界检查。")

窗口 30 交易日通过：预热、归一化、逐日参考、未来扰动、截断不变、目标日缺失及边界检查。


窗口 60 交易日通过：预热、归一化、逐日参考、未来扰动、截断不变、目标日缺失及边界检查。


窗口 120 交易日通过：预热、归一化、逐日参考、未来扰动、截断不变、目标日缺失及边界检查。


窗口 360 交易日通过：预热、归一化、逐日参考、未来扰动、截断不变、目标日缺失及边界检查。


## 6. 分窗口保存结果

每次从头运行分别覆盖本目录 `im_bpv_seasonality_1m_30td.parquet`、`im_bpv_seasonality_1m_60td.parquet`、`im_bpv_seasonality_1m_120td.parquet`、`im_bpv_seasonality_1m_360td.parquet`；`td` 表示交易日，不是自然日。原来的无窗口单文件已经废弃，不再生成。

四个文件保留相同的完整日期和分钟网格，但各自的预热期和可用值不同。字段及 metadata 显式标记交易日窗口，metadata 同时保存共同的源文件摘要，便于可视化确认是同批数据。

统计值为 float64，NaN 转为 Arrow null。每个文件保存后逐值复读。文件只含分钟结果，不保存当天尚不可用的 $B_d$ 或 $u_d$ 作为特征。

运行 `02_02_im_bpv_window_comparison.ipynb` 查看曲线、时间变化与去季节效果。比较仅描述窗口差异，不根据测试集挑选窗口；若需要参数选择，须先固定训练/验证/测试边界。


In [8]:

export_summary_rows = []
for lookback in lookbacks:
    output_df = output_by_lookback[lookback]
    output_schema = output_schema_by_lookback[lookback]
    for field in output_schema:
        if pa.types.is_floating(field.type):
            assert not np.isinf(output_df[field.name].to_numpy(dtype="float64")).any(), field.name
    output_table = pandas_to_arrow(output_df, output_schema).replace_schema_metadata(output_schema.metadata)
    output_path = research_dir / f"im_bpv_seasonality_1m_{lookback}td.parquet"
    pq.write_table(output_table, output_path, compression="zstd")
    verified_output_table = pq.read_table(output_path)
    assert verified_output_table.schema.equals(output_schema, check_metadata=True)
    assert validate_arrow_table(verified_output_table, output_schema).equals(output_table)
    valid_curve_dates = seasonality_by_lookback[lookback].index[seasonality_by_lookback[lookback].notna().any(axis=1)]
    export_summary_rows.append({
        "lookback_trading_days": lookback, "rows": len(output_df),
        "valid_factor_rows": int(output_df.bpv_seasonality.notna().sum()),
        "valid_curve_days": len(valid_curve_dates),
        "first_valid_date": valid_curve_dates[0] if len(valid_curve_dates) else None,
        "file": output_path.name,
    })
    print(f"已保存并复读: {output_path.name}")
export_summary_df = pd.DataFrame(export_summary_rows)
display(export_summary_df)
print("数据范围:", trade_dates[0], "至", trade_dates[-1])


已保存并复读: im_bpv_seasonality_1m_30td.parquet
已保存并复读: im_bpv_seasonality_1m_60td.parquet


已保存并复读: im_bpv_seasonality_1m_120td.parquet
已保存并复读: im_bpv_seasonality_1m_360td.parquet


,lookback_trading_days,rows,valid_factor_rows,valid_curve_days,first_valid_date,file
0,30,238800,227740,965,2022-09-05,im_bpv_seasonality_1m_30td.parquet
1,60,238800,220660,935,2022-10-25,im_bpv_seasonality_1m_60td.parquet
2,120,238800,206500,875,2023-01-18,im_bpv_seasonality_1m_120td.parquet
3,360,238800,149860,635,2024-01-15,im_bpv_seasonality_1m_360td.parquet


数据范围: 2022-07-25 至 2026-08-28
